## Nuclear segmentation proofreading and classification

#### Elizabeth Finn
#### 1/31/2025

This notebook will do the following: 
1) Read in a single 3D czi image
2) Segment nuclei based on the DAPI channel
3) Iteratively allow the user to classify each cell into four categories: Bad (skipped), Mitotic, Early Prophase, Mid Prophase, and Late Prophase
4) Output each cell with appropriate metadata in the file name

In [ ]:
import re
from aicsimageio import AICSImage
from aicsimageio.writers import OmeTiffWriter
import skimage
from pathlib import Path
from random import sample
import numpy
import random
from matplotlib import pyplot
import matplotlib.patches as patches
from scipy import ndimage
from cellpose import core, utils, io, models, metrics
import ipywidgets
import os

This chunk creates a class of "NuclearMask" objects to use later for outputting nuclear masks.

In [ ]:
class NuclearMask:
  def __init__(self, mask, offset):
    self.mask = mask
    self.offset = offset

  @classmethod
  def build(cls, masks, regionprops):
    (min_z, min_row, min_col, max_z, max_row, max_col) = regionprops.bbox
    offset = (min_z, min_row, min_col)
    mask = masks[min_z:max_z, min_row:max_row, min_col:max_col] == regionprops.label
    return cls(mask, offset)

This chunk will read in a single image and make the appropriate output directories for it.

In [ ]:
image_filename = "sample_img/CS4_g1.czi"
parent = "sample_img"
date = "20250206"
field = "CS4_g1" #This should not contain '.czi' -- just CS2_g1 for instance.

MIPS_directory = "%s/MIPS/%s/%s"%(parent, date, field)
masks_directory = "%s/masks/%s/%s"%(parent,date, field)
dts_directory = "%s/dts/%s/%s"%(parent,date, field)

os.makedirs(MIPS_directory)
os.makedirs(masks_directory)
os.makedirs(dts_directory)

img = AICSImage(image_filename)
fields = len(img.scenes)
channels = img.dims.C
height = img.dims.Z
times = img.dims.T
x = img.dims.X
y = img.dims.Y
DAPI_CHANNEL = 0

cropped_height = min(height, 70)

print("I found %i scenes, %i channels, and %i z-slices and %s timepoints of size %i by %i"%(fields, channels, height, times, x, y))

stack_DAPI = img.get_image_data("ZYX", C = DAPI_CHANNEL)

#### Pre-processing of the DAPI channel

We pre-process the DAPI channel in two ways: 
1) Cropping to the top of the z-stack, as imaging 70 slices in reduces signal and yeilds poorer results
2) Binning each z-slice 3x3, as this resolution worked best for the pre-trained cellpose model baseline.

In [ ]:
cropped_stack = stack_DAPI[0:cropped_height,:,:]
binned_stack = skimage.transform.downscale_local_mean(cropped_stack, (1,3,3))

print(numpy.shape(binned_stack))

This chunk will display five z-slices so we can check our work.

In [ ]:
fig, axs = pyplot.subplots(1,5)

step = cropped_height//5

for i in range(0,min(5, cropped_height)):
    ax = axs[i]
    ax.imshow(binned_stack[i*step])
    
fig.set_size_inches(20,4)
fig.canvas.draw()

Load in our custom model and run it in 3D. 

In [ ]:
model = models.CellposeModel(pretrained_model = 'CP_2025011_meiotic')

masks, flows, styles = model.eval(binned_stack, do_3D = True, anisotropy=2, z_axis=0)

print(numpy.shape(masks))

Display segmented masks for a reality check; only a few mask values can be displayed so use modulus to loop the values specifically for display.

In [ ]:
## Display the segmentation result and cell counts:
labels = numpy.max(masks)
print(labels)

step = cropped_height//3

masks_fordisplay = numpy.mod(masks, 100)

fig, axs = pyplot.subplots(1,3)

for i in range(min(3, cropped_height)):
    ax = axs[i]
    ax.imshow(binned_stack[i*step], cmap="Greys")
    if labels > 1:
        ax.contourf(masks_fordisplay[i*step], levels=range(1,100), alpha=0.2)
    else:
        ax.text(0.0, 1.0, "No Cells",
            fontsize='medium', verticalalignment='top')
   
fig.set_size_inches(20,4)
fig.canvas.draw()

Resize the masks to match the initial image, and pull out individual ones with regionprops 

In [ ]:
masks_dilated = skimage.segmentation.expand_labels(masks, 3)
masks_resized = skimage.transform.resize(masks_dilated, (cropped_height,y,x), preserve_range = True, order=0)

#### Crop and display

Currently: Plot 12 random cells for a reality check. Red outline is nuclear perimeter as segmented by CellPose. Channel order goes DAPI, C1, C2, C3 from top to bottom.

In [ ]:
props = skimage.measure.regionprops(masks_resized)

print("Filtering on area")
props_size_filtered = [prop for prop in props if prop.area > 1000]
print("Filtering on height")
props_z_filtered = [prop for prop in props_size_filtered if prop.bbox[3]-prop.bbox[0] > 1]
print("Filtering on xy")
props_dims_filtered = [prop for prop in props_z_filtered if min([prop.bbox[5]-prop.bbox[2], prop.bbox[4]-prop.bbox[1]]) > 25]

props_filtered = [prop for prop in props_dims_filtered if prop.solidity > 0.6]
print("Total nuclei count: %i\nArea filtered nuclei count: %i\nFinal filtered nuclei count:%i"%(len(props), len(props_size_filtered), len(props_filtered))) 

In [ ]:
stack_full = img.get_image_data("CZYX")
count = 10
props_selected = sample(props_filtered, count)
fig, axs = pyplot.subplots(channels+1,count)
i = 0 

for prop in props_selected:
    minz, miny, minx, maxz, maxy, maxx = prop.bbox
    crop = stack_full[:,minz:maxz, max(miny-3, 0):min(maxy+3, 2048), max(0,minx-3):min(2048, maxx+3)]
    crop_mask = numpy.equal(masks_resized[minz:maxz, max(miny-3, 0):min(maxy+3, 2048), max(0,minx-3):min(2048, maxx+3)], prop.label)
    crop_final = (crop + crop*crop_mask)/2
    crop_mip = numpy.amax(crop_final, 1)
    for j in range(channels):
        ax = axs[j,i]
        ax.imshow(crop_mip[j,:,:])
        ax.contour(crop_mask[(maxz-minz)//2,:,:], colors = ("grey"))
    distance_transform = ndimage.distance_transform_edt(numpy.amax(crop_mask, 0))
    normed_transform = distance_transform/numpy.amax(distance_transform)
    dt = 1-normed_transform
    ax = axs[channels,i]
    ax.imshow(dt)
    i = i + 1
    
fig.set_size_inches(15,5)
fig.canvas.draw()

Optional save figure:

In [ ]:
fig.savefig('image_15x5.pdf')

## Set up functions to help with display and output

In [ ]:
i = 0
stack_full = img.get_image_data("CZYX")

In [ ]:
def display_current_cell(n):
    prop = props_filtered[n]
    minz, miny, minx, maxz, maxy, maxx = prop.bbox
    zrange = maxz-minz
    xrange = maxx-minx
    yrange = maxy-miny
    step = max(1, zrange//3)
    
    crop = stack_full[:,minz:maxz, max(miny-3, 0):min(maxy+3, 2048), max(0,minx-3):min(2048, maxx+3)]
    crop_mask = numpy.equal(masks_resized[minz:maxz, max(miny-3, 0):min(maxy+3, 2048), max(0,minx-3):min(2048, maxx+3)], prop.label)
    crop_final = (crop + crop*crop_mask)/2
    crop_mip = numpy.amax(crop_final, 1)

    fig1, axs1 = pyplot.subplots(1,channels)

    for j in range(channels):
        ax = axs1[j]
        ax.imshow(crop_mip[j,:,:])
        ax.contour(crop_mask[(maxz-minz)//2,:,:], colors = ("grey"))
    
    fig1.set_size_inches(12,3)
    fig1.canvas.draw()
    
    fig2, axs2 = pyplot.subplots(1,3)

    for m in range(min(3, zrange)):
        ax = axs2[m]
        ax.imshow(stack_full[0,minz+(m*step)])
        box = patches.Rectangle((minx-5,miny-5), xrange+5, yrange+5, linewidth=1, edgecolor='r', facecolor='none')
        ax.add_patch(box)
   
    fig2.set_size_inches(27,9)
    fig2.canvas.draw()

In [ ]:
def output_current_cell(n, cellstage):
    #load nuclear mask
    prop = props_filtered[n]
    minz, miny, minx, maxz, maxy, maxx = prop.bbox
    crop = stack_full[:,minz:maxz, max(miny-3, 0):min(maxy+3, 2048), max(0,minx-3):min(2048, maxx+3)]
    crop_mask = numpy.equal(masks_resized[minz:maxz, max(miny-3, 0):min(maxy+3, 2048), max(0,minx-3):min(2048, maxx+3)], prop.label)
    
    # output mask to masks directory
    nuclear_mask = NuclearMask.build(masks_resized, prop)
    numpy.save("%s/%s_chXX_nucleus_%04i.npy"%(masks_directory, cellstage, i), nuclear_mask)
    
    # output distance transform to dts directory
    distance_transform = ndimage.distance_transform_edt(numpy.amax(crop_mask, 0))
    normed_transform = distance_transform/numpy.amax(distance_transform)
    dt = 1-normed_transform
    numpy.save("%s/%s_chXX_dt_%04i.npy"%(dts_directory, cellstage, i), dt)
    
    # output channel MIPS, summed intensities, and z centers to MIPS directory
    crop_final = numpy.empty_like(crop)
    crop_masked = numpy.empty_like(crop)

    for j in range(0,channels):
        crop_masked[j] = crop[j]*crop_mask
        crop_final[j] = numpy.ndarray.round((crop_masked[j]/numpy.amax(crop_masked[j]))*65535)
        MIP = numpy.amax(crop_final[j], axis=0)
        numpy.save("%s/%s_ch%s_maximum_projection_nucleus_%04i.npy"%(MIPS_directory, cellstage, j+1, i), MIP)
        summed = numpy.sum(crop_final[j], axis=0) + 0.01
        numpy.save("%s/%s_ch%s_summed_intensity_nucleus_%04i.npy"%(MIPS_directory, cellstage, j+1, i), summed)
        rearranged_z = crop_final[j].transpose(1,2,0)
        zcenter = numpy.dot(rearranged_z, range(crop_final[j].shape[0]))/summed
        numpy.save("%s/%s_ch%s_z_center_nucleus_%04i.npy"%(MIPS_directory, cellstage, j+1, i), zcenter)
            

## Interactive chunk! Run this once per cell to be sorted.

The below chunk will make a drop-down menu with cell types, and a button to save the cell data, displayed above the cell to be sorted. Pick out the type and press save, then run the chunk again to get a new cell.

If you press save accidentally, make a note of the save message ("Saved cell X with value Y") so that I can go in and remove those cells from the training set later -- and then pick the right option in the drop-down and save again. It won't over-write an old classification with a correct one.

If the cell is poorly segmented and you don't want to save it, don't click save and just run the chunk again.

In [ ]:
cellstage_menu = ipywidgets.Dropdown(
    options=[('Early Prophase', 'EP'), 
             ('Mid Prophase', 'MP'), 
             ('Late Prophase', 'LP'),
             ('Spermatogonia', 'SG'),
             ('Leydig', 'LD'),
             ('Sertoli', 'ST')],
    value='EP',
    description='Cell Type:',
)

output_button = ipywidgets.Button(description="Save cell")
output = ipywidgets.Output()

display_current_cell(i)
display(ipywidgets.HBox([cellstage_menu, output_button, output]))

def output_button_clicked(self): 
    global i
    output_current_cell(i, cellstage_menu.value)
    with output:
        output.clear_output()
        print("Saved cell %i with value %s"%(i, cellstage_menu.value))

output_button.on_click(output_button_clicked)

i = i + 1

Keep repeating the chunk above until you run out of cells. 